# 02 · Source grouping and leakage-free split
> **Research prototype.** Predicts *dataset-defined* dementia categories from preprocessed JPG brain MRI slices. Not a clinically validated diagnostic system.

Steps 12–16: build `source_group_id` (patient ID > study ID > exam ID > original-image link > exact duplicate >
validated perceptual transform), create a new grouped stratified 70/15/15 split, run the leakage assertions (training
is blocked if any fails), save `metadata.csv`, `train.csv`, `val.csv`, `test.csv`, and analyse RGB vs grayscale and
superficial shortcuts.

In [ ]:
# Bootstrap: locate the project code (local checkout, /kaggle/working, or a Kaggle dataset upload)
import os, sys, glob, shutil, json
from pathlib import Path

CANDIDATES = [Path.cwd().parent, Path.cwd(), Path("/kaggle/working/dementia_mri_project")]
CANDIDATES += [Path(p).parent for p in glob.glob("/kaggle/input/*/dementia_mri_project/src")
               + glob.glob("/kaggle/input/*/*/dementia_mri_project/src") + glob.glob("/kaggle/input/*/src")]
CODE = next((c for c in CANDIDATES if (c / "src" / "run_pipeline.py").exists()), None)
if CODE is None:
    raise FileNotFoundError("Project code not found. On Kaggle: upload dementia_mri_project.zip as a Dataset and add it "
                            "as an Input (Kaggle extracts it), or put the project folder next to this notebook.")
if str(CODE).startswith("/kaggle/input"):                       # /kaggle/input is read-only -> work in /kaggle/working
    dst = Path("/kaggle/working/dementia_mri_project")
    shutil.copytree(CODE / "src", dst / "src", dirs_exist_ok=True)
    shutil.copy(CODE / "predict.py", dst / "predict.py")
    if not (dst / "configs" / "config.json").exists():
        shutil.copytree(CODE / "configs", dst / "configs", dirs_exist_ok=True)
    for sub in ("data", "checkpoints", "results", "gradcam", "reports"):   # outputs of earlier notebooks, if attached
        if (CODE / sub).exists() and not (dst / sub).exists():
            shutil.copytree(CODE / sub, dst / sub)
    CODE = dst
sys.path.insert(0, str(CODE)); os.chdir(CODE)
print("Project code:", CODE)

from src.utils import load_config, get_paths, print_env, load_json, save_json
from src.nb_helpers import show, show_md, show_table, sample_grid

CONFIG_OVERRIDES = {}      # e.g. {"batch_size": 16, "num_workers": 4, "epochs_stage2": 30}
cfg = load_config(overrides=CONFIG_OVERRIDES)
paths = get_paths(cfg)
ENV = print_env()
print("Outputs ->", paths["root"])

In [ ]:
from src.run_pipeline import stage_split
META, SPLIT = stage_split(cfg, paths)
print(SPLIT["patient_level_statement"])

## Leakage assertions (re-checked here from the saved CSV files)

In [ ]:
import pandas as pd
tr, va, te = (pd.read_csv(paths["data"] / f"{s}.csv") for s in ("train", "val", "test"))
train_groups, val_groups, test_groups = set(tr.source_group_id), set(va.source_group_id), set(te.source_group_id)
for name, a, b in (("train/val", train_groups, val_groups), ("train/test", train_groups, test_groups), ("val/test", val_groups, test_groups)):
    if not a.isdisjoint(b):
        print("OVERLAPPING GROUPS", name, sorted(a & b)[:20])
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)
assert set(tr.sha256).isdisjoint(set(te.sha256)) and set(tr.sha256).isdisjoint(set(va.sha256)) and set(va.sha256).isdisjoint(set(te.sha256))
print("All group and hash assertions passed:", len(tr), len(va), len(te))

## Split distribution

In [ ]:
show(paths["results"] / "split_distribution.png")
show(paths["results"] / "split_class_distribution.png")
display(META.groupby(["split", "grouping_method"]).size().unstack(fill_value=0))
display(META.groupby("split").agg(images=("filepath", "size"), groups=("source_group_id", "nunique")))

## RGB vs grayscale, brain crop, shortcut analysis

In [ ]:
print(json.dumps(SPLIT["preprocessing"], indent=1))
sc = load_json(paths["results"] / "shortcut_analysis.json")
print("Validation macro-F1 from image statistics only:", sc.get("val_macro_f1_from_image_statistics"))
print(sc.get("interpretation"))
display(pd.DataFrame(sc["per_class_means"]).T)
print("Top features:", list(sc.get("feature_importance", {}).items())[:8])

## Leakage report

In [ ]:
show_md(paths["reports"] / "leakage_report.md")